# VidBest — Colab-only, genuinely generated 2D animation (manual free-GPU test)

**සිංහල:** මෙය **Colab තුළම** ඔබ විසින් run කරන interactive GPU notebook එකකි. GitHub CPU fallback වීඩියෝව නැවත copy කිරීමක් නොවේ. AnimateDiff-Lightning විසින් අලුත් frames ජනනය කරන විට පමණක් `colab_ai_new_clip.mp4` හදයි.

1. Colab menu → **Runtime → Change runtime type → T4 GPU** (if available).
2. Run **GPU verification** cell; it does not download model weights.
3. Read model licence / usage notice. To generate a short **new AI motion clip**, change `RUN_AI_VIDEO = False` to `True` and run subsequent cells.
4. The model downloads several GB. GPU memory, Colab limits, and quality are **not guaranteed**. Only use your own prompt/media and comply with model restrictions.
5. Download the result before disconnecting. This notebook does **not** use Kaggle, Oracle, R2, Cloudflare tunnels, unattended workers, AI Horde keys or GitHub Secrets.

**No AI video has yet been verified from this notebook**. Only inspect/run the cells on a GPU Colab session. This first test generates roughly a one-second silent motion clip, not a full natural 23-second episode.

Sources: [AnimateDiff-Lightning](https://huggingface.co/ByteDance/AnimateDiff-Lightning), [Diffusers docs](https://huggingface.co/docs/diffusers/api/pipelines/animatediff), [Colab FAQ](https://research.google.com/colaboratory/faq.html). AnimateDiff-Lightning weights are CreativeML OpenRAIL-M; check the base checkpoint's terms independently before commercial use.

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU available. In Colab choose Runtime > Change runtime type > T4 GPU, then reconnect.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
# This probes real GPU computation but is NOT AI video generation.
tensor = torch.ones((256, 256), device="cuda")
assert (tensor * 2).sum().item() == 131072
print("Colab GPU arithmetic verified.")

## 1. Your original cartoon prompt

This is **text-to-video**, not a reference-image animation importer. Character consistency across independent generations is not guaranteed. Keep the same authored character/rig in Snap2D for most shots; reserve Colab for unusual motion.

**To minimize free GPU usage, leave RUN_AI_VIDEO=False until you are ready.**

In [ ]:
RUN_AI_VIDEO = False   # Change to True ONLY to download weights and generate AI frames
PROMPT = (
  "Original funny 2D hand-drawn comedy, one lanky stick-figure cartoon boy with oversized "
  "expressive eyes and thick black ink outlines suddenly notices an empty cookie jar, "
  "jumps back in surprise, exaggerated reaction pose, dynamic smear movement, "
  "muted olive-grey plain background, clean flat colors, simple cartoon animation"
)
FRAMES = 8       # Small first test; can increase to 12 or 16 after successful test
WIDTH = 320
HEIGHT = 320
FPS = 8
SEED = 2317
assert FRAMES in (8,12,16)
assert WIDTH % 8 == HEIGHT % 8 == 0
print("Configured. AI generation enabled:",RUN_AI_VIDEO)

## 2. Install the open-source Diffusers runtime (optional)

Only runs when you select AI generation. Model weights and base checkpoint can require several GB of disk. GPU allocation must already be active.

This uses ByteDance's documented **4-step AnimateDiff-Lightning motion adapter** and the ToonYou SD1.5-style base. The models have their own licenses; do not assume they're MIT.

In [ ]:
if RUN_AI_VIDEO:
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "diffusers==0.35.2", "transformers>=4.44,<5",
        "accelerate>=1,<2", "safetensors>=0.4,<1",
        "huggingface_hub>=0.26,<2", "imageio[ffmpeg]>=2.34"
    ], check=True)
    print("Dependencies installed. Downloading model in next cell.")
else:
    print("Skipped install: RUN_AI_VIDEO is False")

## 3. Generate a NEW AI video clip (GPU)

This is the stage which actually runs model inference. It is not a CPU fallback or a static montage. If out-of-memory occurs, restart the GPU runtime and keep the first test to 8 frames at 320×320. Success and time cannot be guaranteed.

In [ ]:
if RUN_AI_VIDEO:
    import gc, time
    from pathlib import Path
    from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
    from diffusers.utils import export_to_video
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    t0 = time.monotonic()
    ckpt = hf_hub_download("ByteDance/AnimateDiff-Lightning",
                           "animatediff_lightning_4step_diffusers.safetensors")
    adapter = MotionAdapter()
    adapter.load_state_dict(load_file(ckpt, device="cpu"))
    adapter = adapter.to(dtype=torch.float16)
    pipe = AnimateDiffPipeline.from_pretrained(
        "frankjoshua/toonyou_beta6",
        motion_adapter=adapter,
        torch_dtype=torch.float16
    )
    pipe.scheduler = EulerDiscreteScheduler.from_config(
        pipe.scheduler.config, timestep_spacing="trailing", beta_schedule="linear")
    pipe.enable_vae_slicing()
    pipe.enable_model_cpu_offload()  # lower VRAM use; may be slower
    result = pipe(
        PROMPT, negative_prompt="watermark, text, bad anatomy, blurry, photo",
        num_inference_steps=4, guidance_scale=1.0, num_frames=FRAMES,
        width=WIDTH, height=HEIGHT,
        generator=torch.Generator(device="cpu").manual_seed(SEED),
    )
    video_file = Path("/content/colab_ai_new_clip.mp4")
    export_to_video(result.frames[0], str(video_file), fps=FPS)
    assert video_file.stat().st_size > 1000
    print("NEW AI VIDEO COMPLETED:",video_file, video_file.stat().st_size, "bytes")
    print("Inference and model loading elapsed seconds:",round(time.monotonic()-t0,1))
    del pipe, result, adapter
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("No AI generation attempted. This notebook does not claim an AI video exists.")

## 4. Play and download your NEW AI clip

Only if it was generated in the previous cell. Free Colab sessions can reset; download before disconnecting.

In [ ]:
from pathlib import Path
from IPython.display import display, Video
video_file=Path("/content/colab_ai_new_clip.mp4")
if RUN_AI_VIDEO and video_file.exists():
    display(Video(str(video_file),embed=True))
    from google.colab import files
    files.download(str(video_file))
else:
    print("No new GPU AI MP4 yet. Enable RUN_AI_VIDEO and run generation first.")